# Week 09 (training): Conditioning the diffusion model

In Week 08 you built and trained the unconditional diffusion model — a
network that learns the *marginal* distribution of residuals across all
training cycles, samples from it, and is ablated to confirm the timestep
embedding earns its keep. This week adds the missing ingredient:
**conditioning** on each window's smoothed sunspot area `area_smoothed`
and universal-path latitude `mu_universal`. With those two scalars
threaded into the network, the model goes from "produces plausible
residuals on average" to "produces residuals targeted at the specific
window the caller is asking about."

The architectural change is small. The Dataset returns a dictionary with keys `{"r_clean", "cond"}`
where cond is the 2-vector `(area_smoothed, mu_universal)`. The MLP gets a
slightly wider input — `r_t` and the timestep embedding *and* the
conditioning vector all concatenated at layer zero. The LightningModule's
training step unpacks the tuple and passes `cond` through. The sampler
accepts a per-sample conditioning vector and threads it through every
reverse step. None of the diffusion-specific machinery from Week 08 —
the schedule, the forward equation, the ε-prediction objective, the
timestep embedding itself — changes.

**By the end of this notebook you should have:**
- A trained conditional diffusion model that learns p(r | area, mu) — saved
  as `ckpt_conditional.ckpt`.
- A sanity-checked architecture confirming that t-sensitivity *and*
  cond-sensitivity both work: holding r_t fixed and varying t changes the
  output, *and* holding r_t and t fixed and varying cond also changes the
  output. The conditioning ablation built into Week 08 (the
  `use_timestep_embedding=False` flag) has a direct conditioning analogue
  here: if your model is somehow ignoring `cond`, the cond-sensitivity
  check is the only thing that will catch it before the evaluation
  notebook's distributional comparisons silently lie.

In [ ]:
import os, subprocess, sys

# Keep this path if working in Colab
# repo_path = "/content/butterflai"

# Use this path if working locally
repo_path = "../../"

if not os.path.isdir(repo_path):
    subprocess.run(["git", "clone", "https://github.com/SwRI-IDEA-Lab/butterflai.git", repo_path], check=True)
else:
    try:
        subprocess.run(["git", "-C", repo_path, "pull"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"git pull skipped: {e}")
sys.path.insert(0, repo_path)
from infrastructure.utils.colab_setup import setup
setup()


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
from einops import repeat

import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger, CSVLogger
import wandb


In [ ]:
# ── locate Week 08/09 artifacts (split across two folders) ─────────────────
# The week-08 and week-09 deliverables are spread across weeks/week_08/
# and weeks/week_09/. We search both for each artifact independently so the
# notebook works regardless of where any given file sits.
def _find(filename, search_dirs):
    for d in search_dirs:
        p = os.path.join(d, filename)
        if os.path.isfile(p):
            return p
    return None

_cwd = os.getcwd()
_search_dirs = []
for _base in [_cwd] + [os.path.abspath(os.path.join(_cwd, *[".."] * i)) for i in range(0, 5)]:
    for _sub in [("weeks", "week_09"), ("weeks", "week_08")]:
        _candidate = os.path.join(_base, *_sub)
        if os.path.isdir(_candidate) and _candidate not in _search_dirs:
            _search_dirs.append(_candidate)
    if ("week_08" in _base or "week_09" in _base) and os.path.isdir(_base) and _base not in _search_dirs:
        _search_dirs.append(_base)

_unconditioned_py  = _find("unconditioned_infrastructure.py", _search_dirs)
_conditioned_py    = _find("conditioned_infrastructure.py",   _search_dirs)
_parquet_path      = _find("diffusion_windows.parquet", _search_dirs)
_classical_py      = _find("butterflAI_model.py",     _search_dirs)
_classical_weights = _find("official_model.npz",      _search_dirs)

_missing = [n for n, p in [
    ("unconditioned_infrastructure.py", _unconditioned_py),
    ("conditioned_infrastructure.py",   _conditioned_py),
    ("diffusion_windows.parquet", _parquet_path),
    ("butterflAI_model.py",     _classical_py),
    ("official_model.npz",      _classical_weights),
] if p is None]
if _missing:
    raise FileNotFoundError(
        f"Cannot locate {_missing} under weeks/week_08 or weeks/week_09. "
        f"Searched: {_search_dirs}"
    )

_repo_root = os.path.abspath(os.path.join(os.path.dirname(_conditioned_py), "..", ".."))
for _p in [_repo_root,
           os.path.dirname(_unconditioned_py),
           os.path.dirname(_conditioned_py),
           os.path.dirname(_classical_py)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# ── import: Week 08 reused machinery + Week 09 conditional templates ───────
from unconditioned_infrastructure import (
    make_cosine_schedule,
    sinusoidal_embedding, TimestepEmbedding,
    SampleQualityCallback,
)
from conditioned_infrastructure import (
    ConditionalResidualDataset,
    ConditionalDiffusionMLP,
    ConditionalDiffusionLightning,
    sample_conditional,
)

# ── load Week 07 outputs ───────────────────────────────────────────────────
windows_df = pd.read_parquet(_parquet_path)

LAT_BINS    = np.linspace(0, 45, 16)
BIN_WIDTH   = 3.0
BIN_CENTERS = 0.5 * (LAT_BINS[:-1] + LAT_BINS[1:])

# ── classical ButterflAI model (kept in scope for ad-hoc inspection) ───────
from butterflAI_model import ButterflAIModel
classical = ButterflAIModel(_classical_weights)

# ── cosine schedule (single source of truth from the Week 08 script) ───────
T = 200
alpha_np, sigma_np, _alpha_bar_np = make_cosine_schedule(T=T, s=0.008)

print(f"Loaded {len(windows_df)} windows from diffusion_windows.parquet")
print(f"  splits  : {windows_df['split'].value_counts().sort_index().to_dict()}")
print(f"  cycles  : {sorted(windows_df['cycle'].unique())}")
print(f"  cond raw ranges (train split):")
_train = windows_df[windows_df['split'] == 'train']
for col in ['area_smoothed', 'mu_universal']:
    print(f"    {col:14s}: mean={_train[col].mean():.3f}, std={_train[col].std():.3f}, "
          f"min={_train[col].min():.3f}, max={_train[col].max():.3f}")
print(f"  schedule: T={T}, arrays length {len(alpha_np)}")
print(classical)

---
## Task 45 — `ConditionalResidualDataset` (template in `conditioned_infrastructure.py`)

The new Dataset returns a **dict** instead of a tuple:

```
{"r_clean": float32 tensor (15,),
 "cond":    float32 tensor (2,)}
```

Dict-returning Datasets are the convention worth adopting from Week 09
onward. Dicts are self-documenting at access time (`batch["cond"]` needs
no memory of position), extensible (adding a `cycle_phase` field later
does not break unpacking sites in the training step, the sampler, or the
evaluation notebook — only the consumers that actually use the new field
need updating), and PyTorch's default DataLoader collation handles
dict-of-tensors natively. The tuple-returning version of this dataset
would work too, but every new conditioning variable would require
chasing down every `r, cond = batch` line in the codebase. The dict
pattern scales; the tuple pattern does not.

Two pieces of standardization are still happening, exactly as discussed
in the script's docstring:

**Residuals** are standardized per-bin to unit variance, exactly as in
Week 08. Each split computes its own `bin_means` / `bin_stds`. The
LightningModule persists the train-split bin statistics as buffers so the
sampler can de-standardize at inference time.

**Conditioning** is standardized using *train-split-only* statistics —
this is the part that requires care. The network learns to expect cond
inputs in the train-set's normalized scale. If the validation dataset
standardized using its own statistics, the network would see
distributionally different conditioning at evaluation time than it saw at
training time, which silently corrupts every cond-sensitivity check
downstream.

The template handles this for you: when you construct any
`ConditionalResidualDataset` without passing `cond_means` / `cond_stds`,
the constructor computes them from the *train rows of the supplied
DataFrame* regardless of which split the Dataset itself represents. So
the `val` dataset gets the train-set normalization automatically.

**Open `conditioned_infrastructure.py` and implement:**
- `ConditionalResidualDataset.__init__` (residual standardization +
  conditioning standardization with the train-split-only constraint).
- `ConditionalResidualDataset.__len__`.
- `ConditionalResidualDataset.__getitem__` (returns the dict described
  above with the right keys, dtypes, and shapes).


---
## Task 46 — Visualize the conditional dataset

A dataset class is one of the easier objects to silently misimplement.
For the conditional version there is *one more thing* to check beyond
the Week 08 sanity tests: that conditioning standardization actually uses
train-split statistics for the validation set. If your val dataset has
cond mean far from zero or cond std far from one — but the *train*
dataset does have cond mean zero and cond std one — you accidentally
standardized each split with its own stats, which is the failure mode
this Dataset spec was designed to prevent.

**Tasks:**
- Instantiate `train_dataset` and `val_dataset`. Print their lengths.
  The counts should match the splits printed by the setup cell.
- Pull `train_dataset[0]` and verify it is a **dict** with the right keys
  (`"r_clean"`, `"cond"`), each value a float32 tensor of the expected
  shape, finite. If you accidentally returned a tuple from `__getitem__`,
  this is where you find out.
- Print the `cond_means` and `cond_stds` of both datasets. They must be
  **identical** across train and val. If they differ, the
  train-split-only rule was violated.
- For the train dataset, verify that the cond mean is ≈ 0 and the cond
  std is ≈ 1 (both within ~1e-5). For val, cond mean and std will be
  near these but not exactly — you are applying train statistics to a
  different distribution, so deviations from 0/1 are expected and
  meaningful.
- Plot the raw `(area_smoothed, mu_universal)` joint distribution coloured
  by split, then the normalized joint distribution. The normalized
  version should be centred on (0, 0) with the train cloud reaching
  roughly ±2 in each coordinate.


In [ ]:
# ─────────────────────────────────────────────────────────────
# Task 46 — Visualize the conditional dataset
# ─────────────────────────────────────────────────────────────

import torch

# 1. Instantiate datasets
train_dataset = ConditionalResidualDataset(windows_df, "train")
val_dataset   = ConditionalResidualDataset(
    windows_df,
    "val",
    cond_means=train_dataset.cond_means,
    cond_stds=train_dataset.cond_stds
)

print("\n── Dataset sizes ─────────────────────────────")
print(f"train: {len(train_dataset)}")
print(f"val  : {len(val_dataset)}")

# 2. Check __getitem__ structure
item = train_dataset[0]

print("\n── __getitem__ check ─────────────────────────")
print(f"type: {type(item)}")
print(f"keys: {item.keys()}")

assert isinstance(item, dict)
assert set(item.keys()) == {"r_clean", "cond"}

assert item["r_clean"].shape == (15,)
assert item["cond"].shape == (2,)

assert item["r_clean"].dtype == torch.float32
assert item["cond"].dtype == torch.float32

assert torch.isfinite(item["r_clean"]).all()
assert torch.isfinite(item["cond"]).all()

print("✓ __getitem__ structure OK")

# 3. Compare conditioning stats
print("\n── Conditioning statistics ───────────────────")

print("train cond_means:", train_dataset.cond_means)
print("val   cond_means:", val_dataset.cond_means)

print("train cond_stds :", train_dataset.cond_stds)
print("val   cond_stds :", val_dataset.cond_stds)

assert np.allclose(train_dataset.cond_means, val_dataset.cond_means)
assert np.allclose(train_dataset.cond_stds,  val_dataset.cond_stds)

print("✓ Train/val conditioning stats match")

# 4. Check normalization quality (train vs val)
C_train = train_dataset.all_cond()
C_val   = val_dataset.all_cond()

print("\n── Normalization check ───────────────────────")

print("train mean:", C_train.mean(axis=0))
print("train std :", C_train.std(axis=0))

print("val mean  :", C_val.mean(axis=0))
print("val std   :", C_val.std(axis=0))

# Train should be ~0 mean / ~1 std
assert np.allclose(C_train.mean(axis=0), 0, atol=1e-5)
assert np.allclose(C_train.std(axis=0),  1, atol=1e-5)

print("✓ Train normalization correct")

# Val will deviate slightly (expected)

# 5. Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Raw conditioning space ─────────────────────────────
ax = axes[0]

for split_name, ds in [("train", train_dataset), ("val", val_dataset)]:
    df_split = windows_df[windows_df["split"] == split_name]
    ax.scatter(df_split["area_smoothed"], df_split["mu_universal"])

ax.set_title("Raw conditioning space")
ax.set_xlabel("area_smoothed")
ax.set_ylabel("mu_universal")

# ── Normalized conditioning space ─────────────────────
ax = axes[1]

for ds in [train_dataset, val_dataset]:
    C = ds.all_cond()
    ax.scatter(C[:, 0], C[:, 1])

ax.set_title("Normalized conditioning space")
ax.set_xlabel("area_smoothed (norm)")
ax.set_ylabel("mu_universal (norm)")
ax.axhline(0)
ax.axvline(0)

plt.tight_layout()
plt.show()

print("\n✓ Task 46 complete")

---
## Task 47 — Build the DataLoaders

Same DataLoader pattern as Week 08. PyTorch's default collation handles
the Dataset's dict return automatically: each key in the per-item dict
becomes a key in the per-batch dict, with the leading batch dimension
prepended to every tensor value. So `next(iter(train_loader))` is a
dict:

```
{"r_clean": (B, 15) float32, "cond": (B, 2) float32}
```

Verify this explicitly — the LightningModule's `_shared_step` will read
from these keys, and a tuple-returning Dataset (from accidentally writing
`return r, cond` instead of `return {"r_clean": r, "cond": cond}` in
`__getitem__`) will give batches that look superficially similar but
fail in the training step several layers deeper.


In [ ]:
# ─────────────────────────────────────────────────────────────
# Task 47 — DataLoaders + batch sanity check
# ─────────────────────────────────────────────────────────────

BATCH_SIZE = 64

# 1. Build loaders (same pattern as Week 08)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

# 2. Batch sanity checks
for name, loader in [("train", train_loader), ("val", val_loader)]:
    batch = next(iter(loader))

    print(f"\n── {name} batch check ─────────────────────")

    # Type check (critical: must be dict, not tuple/list)
    assert isinstance(batch, dict), f"expected dict, got {type(batch)}"

    # Key check
    assert set(batch.keys()) == {"r_clean", "cond"}, f"wrong keys: {batch.keys()}"

    r = batch["r_clean"]
    c = batch["cond"]

    # Shape checks
    assert r.ndim == 2 and r.shape[1] == 15
    assert c.ndim == 2 and c.shape[1] == 2

    # dtype checks
    assert r.dtype == torch.float32
    assert c.dtype == torch.float32

    # finiteness checks
    assert torch.isfinite(r).all()
    assert torch.isfinite(c).all()

    print(f"✓ batch type      : {type(batch)}")
    print(f"✓ r_clean shape   : {tuple(r.shape)}")
    print(f"✓ cond shape      : {tuple(c.shape)}")
    print(f"✓ r_clean dtype   : {r.dtype}")
    print(f"✓ cond dtype      : {c.dtype}")

print("\n✓ Task 47 complete — DataLoaders are correct")

---
## Task 48 — `ConditionalDiffusionMLP` (template in `conditioned_infrastructure.py`)

The forward signature changes from `(r_t, t) → eps_hat` to
`(r_t, t, cond) → eps_hat`. The conditioning vector is concatenated
alongside r_t and the timestep embedding at the input layer:

```
x = torch.cat([r_t, t_emb, cond], dim=-1)   # shape (B, 15 + 128 + 2)
```

No new architectural concepts beyond that. The `TimestepEmbedding` is
reused unchanged from Week 08, imported from `unconditioned_infrastructure.py`.

**Open `conditioned_infrastructure.py` and implement** the `__init__` and
`forward` of `ConditionalDiffusionMLP`. Default constructor arguments
are pinned to match what the rest of this notebook expects.

---
## Task 49 — Sanity tests: t-sensitivity *and* cond-sensitivity

Week 08 introduced the t-sensitivity check: hold r_t fixed, vary t,
verify the output measurably changes. That check defends against a
silently-broken timestep embedding.

The conditional model needs the analogous check for the conditioning:
**cond-sensitivity** — hold r_t and t fixed, vary `cond` across the
range of (area, mu) values the model will actually see, verify the output
measurably changes. If it does not, your model is silently unconditional
no matter what the class name says, and every distributional comparison
in the evaluation notebook will give answers that look reasonable but
are not measuring what you think they are.

Run **both** checks on a freshly-initialized model before training. The
checks are inexpensive; the failure mode they catch is exactly the kind
of silent corruption that wastes days of training time.

For the cond-sensitivity check, use a *realistic* range of conditioning
values — sample five (area, mu) points spanning roughly ±2 standard
deviations of the train distribution (the normalized range the model
will actually encounter), not arbitrary synthetic values. This makes the
check both more meaningful and a closer analogue of the t-sensitivity
check, which uses real timestep values.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Task 49 — t-sensitivity + cond-sensitivity checks
# ─────────────────────────────────────────────────────────────

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from einops import repeat

# ---- Fresh model (CRITICAL: untrained) ----
torch.manual_seed(1)
model_test = ConditionalDiffusionMLP()
model_test.eval()

T_local = 200  # must match setup cell

# ─────────────────────────────────────────────
# 1. t-sensitivity check
# ─────────────────────────────────────────────

r_t_fixed = torch.randn(15)
cond_zero  = torch.zeros(2)

t_values = torch.tensor(
    [0, T_local // 4, T_local // 2, 3 * T_local // 4, T_local - 1],
    dtype=torch.long
)

r_t_batch  = repeat(r_t_fixed, "d -> n d", n=5)
cond_batch = repeat(cond_zero, "d -> n d", n=5)

with torch.no_grad():
    out_t = model_test(r_t_batch, t_values, cond_batch)

# pairwise L2 distances
D_t = torch.cdist(out_t, out_t, p=2)

print("\n── t-sensitivity check ───────────────")
print("pairwise distance matrix:\n", D_t)

assert (D_t > 1e-6).any(), "t-sensitivity FAILED: model ignores timestep"

print("✓ t-sensitivity passed")

# ─────────────────────────────────────────────
# 2. cond-sensitivity check
# ─────────────────────────────────────────────

# ±2 range in normalized conditioning space
cond_values = torch.tensor([
    [-2., -2.],
    [-1.,  1.],
    [ 0.,  0.],
    [ 1., -1.],
    [ 2.,  2.]
], dtype=torch.float32)

t_fixed = torch.full((5,), T_local // 2, dtype=torch.long)

r_t_batch = repeat(r_t_fixed, "d -> n d", n=5)

with torch.no_grad():
    out_c = model_test(r_t_batch, t_fixed, cond_values)

D_c = torch.cdist(out_c, out_c, p=2)

print("\n── cond-sensitivity check ─────────────")
print("pairwise distance matrix:\n", D_c)

assert (D_c > 1e-6).any(), "cond-sensitivity FAILED: model ignores conditioning"

print("✓ cond-sensitivity passed")

# ─────────────────────────────────────────────
# 3. Visualization: heatmaps
# ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

im0 = axes[0].imshow(D_t.numpy())
axes[0].set_title("t-sensitivity (pairwise L2)")
axes[0].set_xlabel("t index")
axes[0].set_ylabel("t index")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(D_c.numpy())
axes[1].set_title("cond-sensitivity (pairwise L2)")
axes[1].set_xlabel("cond index")
axes[1].set_ylabel("cond index")
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

print("\n✓ Task 49 complete — both sensitivities verified")

---
## Task 50 — `ConditionalDiffusionLightning` (template in `conditioned_infrastructure.py`)

The only structural changes from Week 08's `DiffusionLightning`:

1. `_shared_step` reads `batch["r_clean"]` and `batch["cond"]` from the
   dict batch (rather than unpacking a tuple) and passes `cond` through
   to `self.model(r_t, t, cond)`.
2. The constructor accepts `cond_means` and `cond_stds` and registers them
   as buffers alongside the existing `bin_means` / `bin_stds`. This keeps
   *all* relevant standardization statistics on the checkpoint, so the
   evaluation notebook can load and re-normalize conditioning without
   needing the training-time dataset.
3. `save_hyperparameters(ignore=[...])` now ignores the four statistics
   tensors alongside `model`, `alpha`, `sigma`.

The `configure_optimizers` method is *identical* to Week 08's. Copy it
from `unconditioned_infrastructure.py`.

**Open `conditioned_infrastructure.py` and implement** `__init__`,
`_shared_step`, and `configure_optimizers`. The `training_step` and
`validation_step` are filled in for you — they call `_shared_step` and
log loss exactly as in Week 08.

---
## Task 51 — Train the conditional model


In [ ]:
# Task 51: Train the conditional diffusion model
# Depends on: ConditionalDiffusionMLP, ConditionalDiffusionLightning,
#             cond_train_loader, cond_val_loader, cds_train, cds_val,
#             alpha_np, sigma_np, T, BIN_CENTERS, BIN_WIDTH, DEVICE

import torch
import numpy as np
import matplotlib.pyplot as plt
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger, CSVLogger
from pytorch_lightning.callbacks import (
    ModelCheckpoint, EarlyStopping, LearningRateMonitor)
import os

# ── 0. Reproducibility ────────────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

# ── 1. De-standardize residuals for SampleQualityCallback ────────────────
# The callback compares generated samples against real residuals.
# Both must be in physical density units (not standardized).
# Read by key since each dataset item is now a dict.

bin_means_np = cds_train.bin_means   # (15,) numpy
bin_stds_np  = cds_train.bin_stds    # (15,) numpy

all_train_std  = np.stack([cds_train[i]["r_clean"].numpy()
                            for i in range(len(cds_train))])   # (N_train, 15)
all_val_std    = np.stack([cds_val[i]["r_clean"].numpy()
                            for i in range(len(cds_val))])     # (N_val, 15)

# De-standardize: r_phys = r_std * bin_stds + bin_means
all_train_phys = all_train_std * bin_stds_np + bin_means_np   # (N_train, 15)
all_val_phys   = all_val_std   * bin_stds_np + bin_means_np   # (N_val, 15)

print(f"Physical residual arrays:")
print(f"  all_train_phys : {all_train_phys.shape}  "
      f"range [{all_train_phys.min():.4f}, {all_train_phys.max():.4f}]")
print(f"  all_val_phys   : {all_val_phys.shape}  "
      f"range [{all_val_phys.min():.4f}, {all_val_phys.max():.4f}]")

# ── 2. ConditionalSampleQualityCallback ───────────────────────────────────
# Subclasses the Week 08 callback, overriding the sampling call to use
# sample_conditional with a fixed reference conditioning batch.

@torch.no_grad()
def sample_conditional(lightning_module, cond_batch, data_dim=15, device="cpu"):
    """
    DDIM sampler for the conditional model.

    Parameters
    ----------
    lightning_module : ConditionalDiffusionLightning
    cond_batch       : torch.Tensor (N, 2) — normalized conditioning vectors
    data_dim         : int
    device           : str

    Returns
    -------
    r_phys : np.ndarray (N, 15) — generated residuals in physical units
    """
    lightning_module.eval()
    lightning_module.to(device)

    alpha = lightning_module.alpha
    sigma = lightning_module.sigma
    T_mod = lightning_module.T
    N     = cond_batch.shape[0]

    cond_batch = cond_batch.to(device)
    r_t        = torch.randn(N, data_dim, device=device)

    for t in range(T_mod - 1, -1, -1):
        t_batch = torch.full((N,), t, dtype=torch.long, device=device)
        eps_hat = lightning_module.model(r_t, t_batch, cond_batch)
        r_0_hat = (r_t - sigma[t] * eps_hat) / alpha[t]
        if t > 0:
            r_t = alpha[t-1] * r_0_hat + sigma[t-1] * eps_hat
        else:
            r_t = r_0_hat

    # De-standardize to physical units
    bin_means = lightning_module.bin_means.cpu().numpy()
    bin_stds  = lightning_module.bin_stds.cpu().numpy()
    r_phys    = r_t.cpu().numpy() * bin_stds + bin_means
    return r_phys


class ConditionalSampleQualityCallback(pl.Callback):
    """
    Periodic distributional quality check for the conditional model.

    At every `every_n_epochs` and at training end, samples N residuals
    using a fixed reference conditioning batch drawn from the training set,
    then computes and logs bin-wise mean MSE and std ratio vs the physical
    training residuals.

    Parameters
    ----------
    train_samples   : np.ndarray (N_train, 15) — physical training residuals
    val_samples     : np.ndarray (N_val, 15)   — physical val residuals
    ref_cond        : torch.Tensor (N_ref, 2)  — normalized reference cond
    every_n_epochs  : int
    n_compare       : int  — number of samples to generate per check
    bin_centers     : np.ndarray (15,)
    bin_width       : float
    device          : str
    """
    def __init__(self, train_samples, val_samples, ref_cond,
                 every_n_epochs=100, n_compare=200,
                 bin_centers=None, bin_width=3.0, device="cpu"):
        super().__init__()
        self.train_samples   = train_samples
        self.val_samples     = val_samples
        self.ref_cond        = ref_cond
        self.every_n_epochs  = every_n_epochs
        self.n_compare       = min(n_compare, len(ref_cond))
        self.bin_centers     = bin_centers if bin_centers is not None \
                               else np.linspace(1.5, 43.5, 15)
        self.bin_width       = bin_width
        self.device          = device
        self.history         = []

    def _run_check(self, pl_module, epoch):
        cond_batch = self.ref_cond[:self.n_compare]
        r_gen      = sample_conditional(
            pl_module, cond_batch,
            data_dim=15, device=self.device)      # (n_compare, 15)

        mean_gen   = r_gen.mean(axis=0)
        mean_train = self.train_samples.mean(axis=0)
        std_gen    = r_gen.std(axis=0)
        std_train  = self.train_samples.std(axis=0)

        mean_mse   = float(np.mean((mean_gen - mean_train)**2))
        std_ratio  = float(std_gen.mean() / max(std_train.mean(), 1e-8))

        self.history.append(dict(
            epoch=epoch, mean_mse=mean_mse, std_ratio=std_ratio))

        pl_module.log("sample_mean_mse",  mean_mse,  prog_bar=False)
        pl_module.log("sample_std_ratio", std_ratio, prog_bar=False)

        print(f"\n  [SampleQuality ep={epoch}]  "
              f"mean_MSE={mean_mse:.6f}  "
              f"std_ratio={std_ratio:.4f}  "
              f"(1.0 = perfect spread)")

    def on_train_epoch_end(self, trainer, pl_module):
        ep = trainer.current_epoch
        if ep > 0 and ep % self.every_n_epochs == 0:
            self._run_check(pl_module, ep)

    def on_train_end(self, trainer, pl_module):
        self._run_check(pl_module, trainer.current_epoch)


# ── 3. Reference conditioning batch for the callback ─────────────────────
# Draw n_compare conditioning vectors from the training set
# (normalized — the sampler will use them as-is)
rng_51    = np.random.default_rng(0)
N_COMPARE = min(200, len(cds_train))
idx_ref   = rng_51.choice(len(cds_train), size=N_COMPARE, replace=False)
ref_cond  = torch.stack(
    [cds_train[int(i)]["cond"] for i in idx_ref])

print(f"Training windows : {len(cds_train)}")
print(f"N_COMPARE set to : {N_COMPARE}")
print(f"ref_cond shape   : {ref_cond.shape}")

# ── 4. Hyperparameters ────────────────────────────────────────────────────
MAX_EPOCHS  = 10_000
LR          = 1e-3
CKPT_COND   = "./ckpt_conditional.ckpt"
PROJECT     = "butterflai-wk09"
RUN_NAME    = "conditional_full"

# ── 5. Model + LightningModule ────────────────────────────────────────────
torch.manual_seed(42)
cond_model = ConditionalDiffusionMLP()

lightning_cond = ConditionalDiffusionLightning(
    model      = cond_model,
    alpha      = alpha_np,
    sigma      = sigma_np,
    T          = T,
    lr         = LR,
    bin_means  = cds_train.bin_means,
    bin_stds   = cds_train.bin_stds,
    cond_means = cds_train.cond_means,
    cond_stds  = cds_train.cond_stds,
)

n_params = sum(p.numel() for p in lightning_cond.parameters())
print(f"\nConditionalDiffusionMLP parameters : {n_params:,}")
print(f"Training windows                   : {len(cds_train)}")
print(f"Val windows                        : {len(cds_val)}")
print(f"Max epochs                         : {MAX_EPOCHS}")

# ── 6. Logger ─────────────────────────────────────────────────────────────
try:
    import wandb
    logger_cond = WandbLogger(
        project   = PROJECT,
        name      = RUN_NAME,
        log_model = False,
        save_dir  = "./wandb_logs",
    )
    logger_cond.log_hyperparams({
        "model"        : "ConditionalDiffusionMLP",
        "n_params"     : n_params,
        "max_epochs"   : MAX_EPOCHS,
        "lr"           : LR,
        "T"            : T,
        "cond_dim"     : 2,
        "data_dim"     : 15,
        "batch_size"   : BATCH_SIZE,
    })
    print("WandB logger ready")
except Exception as e:
    print(f"WandB unavailable ({e}) — using CSVLogger")
    logger_cond = CSVLogger("./csv_logs", name="conditional", version=0)

# ── 7. Callbacks ──────────────────────────────────────────────────────────
ckpt_cb = ModelCheckpoint(
    monitor   = "val_loss",
    mode      = "min",
    save_top_k= 1,
    filename  = "best-{epoch:04d}-{val_loss:.5f}",
    verbose   = False,
)
early_cb = EarlyStopping(
    monitor  = "val_loss",
    patience = 500,       # generous patience for long runs
    mode     = "min",
    verbose  = True,
)
sample_quality_cb = ConditionalSampleQualityCallback(
    train_samples  = all_train_phys,
    val_samples    = all_val_phys,
    ref_cond       = ref_cond,
    every_n_epochs = 500,
    n_compare      = N_COMPARE,
    bin_centers    = LAT_CENTERS,
    bin_width      = BIN_WIDTH,
    device         = DEVICE,
)

# ── 8. Trainer ────────────────────────────────────────────────────────────
trainer_cond = pl.Trainer(
    max_epochs          = MAX_EPOCHS,
    logger              = logger_cond,
    callbacks           = [ckpt_cb, early_cb, sample_quality_cb],
    accelerator         = "auto",
    devices             = "auto",
    log_every_n_steps   = 10,
    enable_progress_bar = True,
    deterministic       = False,
)

# ── 9. Train ──────────────────────────────────────────────────────────────
print(f"\nStarting training — '{RUN_NAME}' in project '{PROJECT}'")
trainer_cond.fit(lightning_cond, cond_train_loader, cond_val_loader)

# ── 10. Save checkpoint ───────────────────────────────────────────────────
trainer_cond.save_checkpoint(CKPT_COND)
print(f"\nCheckpoint saved: {CKPT_COND}")
print(f"Best val_loss   : {ckpt_cb.best_model_score:.6f}")

try:
    import wandb
    wandb.finish()
except Exception:
    pass

# ── 11. Extract and plot loss history ─────────────────────────────────────
def load_loss_history():
    """Try WandB then CSV logger."""
    # CSV path
    for csv_path in [
            "./csv_logs/conditional/version_0/metrics.csv",
            "./csv_logs/version_0/metrics.csv",
    ]:
        if os.path.exists(csv_path):
            import pandas as pd
            metrics = pd.read_csv(csv_path)
            tr = metrics[metrics["train_loss"].notna()]["train_loss"].values
            vl = metrics[metrics["val_loss"].notna()]["val_loss"].values
            return tr, vl
    return np.array([]), np.array([])

train_loss_h, val_loss_h = load_loss_history()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, scale in [(axes[0], "linear"), (axes[1], "log")]:
    plot_fn = ax.plot if scale == "linear" else ax.semilogy
    if len(train_loss_h) > 0:
        plot_fn(train_loss_h, color="tab:blue", linewidth=1.5,
                label="Train loss")
    if len(val_loss_h) > 0:
        plot_fn(val_loss_h, color="tab:orange", linewidth=1.5,
                linestyle="--", label="Val loss")
    if len(val_loss_h) > 0:
        best_ep  = int(np.argmin(val_loss_h))
        best_val = val_loss_h[best_ep]
        ax.axvline(best_ep, color="gray", linewidth=1,
                   linestyle=":", alpha=0.7)
        ax.scatter([best_ep], [best_val], color="tab:red", s=60,
                   zorder=5, label=f"Best val={best_val:.5f}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(f"MSE loss ({scale})")
    ax.set_title(f"Conditional model — {scale} scale")
    ax.legend(fontsize=8)

fig.suptitle(
    f"ConditionalDiffusionMLP training\n"
    f"{n_params:,} params  |  {len(cds_train)} train windows  |  "
    f"T={T}  lr={LR}  max_epochs={MAX_EPOCHS}",
    fontsize=9
)
plt.tight_layout()
plt.show()

# ── 12. Sample quality history plot ───────────────────────────────────────
if len(sample_quality_cb.history) > 0:
    fig2, axes2 = plt.subplots(1, 2, figsize=(12, 4))
    epochs_sq   = [h["epoch"]     for h in sample_quality_cb.history]
    mean_mse_sq = [h["mean_mse"]  for h in sample_quality_cb.history]
    std_ratio_sq= [h["std_ratio"] for h in sample_quality_cb.history]

    axes2[0].plot(epochs_sq, mean_mse_sq, "o-", color="tab:purple",
                  linewidth=2)
    axes2[0].set_xlabel("Epoch"); axes2[0].set_ylabel("Mean MSE")
    axes2[0].set_title("Sample quality: mean MSE vs training\n"
                        "(lower = generated means match training)")

    axes2[1].plot(epochs_sq, std_ratio_sq, "o-", color="tab:green",
                  linewidth=2)
    axes2[1].axhline(1.0, color="black", linewidth=1,
                     linestyle="--", label="Perfect (1.0)")
    axes2[1].set_xlabel("Epoch"); axes2[1].set_ylabel("Std ratio")
    axes2[1].set_title("Sample quality: std ratio (gen/train)\n"
                        "(1.0 = correct spread)")
    axes2[1].legend(fontsize=8)

    fig2.suptitle("Sample quality diagnostics during training",
                  fontsize=9)
    plt.tight_layout()
    plt.show()

# ── 13. Training summary ──────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  Task 51 Training Summary")
print("=" * 60)
print(f"  Epochs trained   : {len(train_loss_h)}")
if len(train_loss_h) > 0:
    print(f"  Final train loss : {train_loss_h[-1]:.6f}")
if len(val_loss_h) > 0:
    print(f"  Final val loss   : {val_loss_h[-1]:.6f}")
    print(f"  Best val loss    : {val_loss_h.min():.6f}  "
          f"(epoch {np.argmin(val_loss_h)})")
    gap = val_loss_h[-1] - train_loss_h[-1] if len(train_loss_h) > 0 else float("nan")
    print(f"  Train/val gap    : {gap:+.6f}  "
          f"({'overfit' if gap > 0.01 else 'ok'})")
print(f"  Checkpoint       : {CKPT_COND}")
if len(sample_quality_cb.history) > 0:
    last_sq = sample_quality_cb.history[-1]
    print(f"  Final mean MSE   : {last_sq['mean_mse']:.8f}")
    print(f"  Final std ratio  : {last_sq['std_ratio']:.4f}")
print("=" * 60)
print(f"\n✓ Task 51 complete — ckpt_conditional.ckpt saved")
print(f"  Ready for Task 52 — conditional sampler")

---
## Handoff to `09d_conditioned_evaluate.ipynb`

You should now have one checkpoint file in this directory:
- `ckpt_conditional.ckpt` (conditional diffusion model)

Together with the Week 08 unconditional checkpoint `ckpt_full.ckpt`, this
gives the evaluation notebook two trained models to compare. The
evaluation notebook will:

- Reload both checkpoints and re-verify their t-sensitivity (and, for the
  conditional model, cond-sensitivity) on the *loaded* state — defends
  against silent state-dict corruption during loading.
- Sample residuals from the conditional model targeted at specific
  validation-window conditioning, visualize them.
- Compare the conditional samples' distribution to held-out validation
  residuals, against the unconditional baseline.

The headline value-add metric — `compute_global_nll` on held-out cycles,
classical alone vs. classical + conditional residuals — is its own
notebook (`09e_diffusion_NLL_evaluation.ipynb`). That's the question
the program has been pointing at since Week 03, and it gets its own
clean space to be answered.

**If you change anything in `conditioned_infrastructure.py` after this notebook
has been run**, the autoreload magic in the evaluation notebook will pick
it up — but checkpoints saved with the old code will not be compatible
with new class signatures. If you change a class signature, retrain.
